# Embedding Ensemble for Rule Violation Detection

This notebook implements a **semantic similarity ensemble** approach for the Jigsaw community rules hackathon.

**Pipeline:**
1. Fine-tune multiple embedding models (GTE-Large, BGE-Large, E5-Large) using triplet loss on rule-violation data
2. For each model: embed test comments and a labeled corpus, then score each test comment by its weighted similarity to violating vs. non-violating examples for the same rule
3. Ensemble the three models' scores via rank-based averaging for a final prediction

**Key idea:** A comment likely violates a rule if it is semantically closer to known violating comments than to non-violating ones for that specific rule.

## Cell 1: Top-level Imports

Minimal imports used directly in this notebook (other modules are imported inside the scripts written below).

In [1]:
import os
import pandas as pd

## Cell 2: `constants.py` — Global Configuration

Writes `constants.py` to disk. Contains all tunable hyper-parameters and path constants:
- `EMBDEDDING_MODEL_PATH`: Path to the base embedding model to fine-tune
- `DATA_PATH`: Location of the Jigsaw dataset (`train.csv`, `test.csv`)
- `EMBEDDING_MODEL_QUERY`: Instruction prefix for instruction-following embedding models (e.g., Qwen3); unused for standard models
- `CLEAN_TEXT`: Whether to normalize/clean comment text before encoding
- `TOP_K`: Number of nearest neighbors to retrieve during semantic search
- `BATCH_SIZE`: Encoding batch size for the SentenceTransformer

In [ ]:
%%writefile constants.py

DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Represent this text for semantic similarity search:"

CLEAN_TEXT = True
TOP_K = 50
BATCH_SIZE = 128

Writing constants.py


## Cell 3: `utils.py` — Data Loading & Preprocessing Utilities

Writes `utils.py` to disk with helper functions shared by all other scripts:
- **`build_prompt(row)`**: Formats a comment row into a model input string (currently uses raw `body` text; the subreddit-prefixed variant is commented out)
- **`cleaner(text)`**: Normalizes text via `cleantext` — fixes unicode/ASCII, strips URLs/emails/phone numbers, replaces them with tokens like `<URL>`
- **`get_dataframe_to_train(data_path)`**: Builds a unified labeled corpus by combining the train split with the positive/negative examples embedded in the test set. Deduplicates on `(body, rule)` to avoid leakage from duplicate entries.
- **`prepare_dataframe(dataframe)`**: Applies prompt formatting and optional cleaning; re-maps `rule_violation` labels to `+1` (violating) / `-1` (non-violating) for the signed similarity scoring used later

In [3]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    #return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""
    return f"""{row["body"]}"""

def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)    
    #dataframe = dataframe.drop_duplicates(ignore_index=True)
    dataframe = dataframe.drop_duplicates(subset=["body", "rule"], ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

Writing utils.py


## Cell 4: `finetune_embedding_model.py` — Triplet Fine-tuning

Writes the fine-tuning script to disk. The `train_embedding_model(base_model_path, save_path)` function:
1. Loads and optionally cleans the training corpus
2. **Builds triplets** of the form `(anchor=rule_text, positive=violating_comment, negative=non_violating_comment)` — each violating comment generates 2 triplets by randomly sampling 2 distinct non-violating comments from the same rule
3. Trains with **TripletLoss** (cosine distance, margin=0.3) for 1 epoch with 10% linear warmup using batch size 8
4. Saves the fine-tuned model and prints a cosine-similarity sanity check: the positive comment should be closer to the anchor (rule) than the negative

The triplet formulation teaches the model that violating comments are semantically close to the rule they break.

In [4]:
%%writefile finetune_embedding_model.py
import pandas as pd

#from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformer, losses, util
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm

from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE,CLEAN_TEXT

import pandas as pd
from sentence_transformers import InputExample
import random
from torch.utils.data import DataLoader
import os
from utils import cleaner

os.environ["WANDB_DISABLED"] = "true"

def train_embedding_model(EMBDEDDING_MODEL_PATH,SAVE_MODEL_PATH):
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    #corpus_dataframe = prepare_dataframe(corpus_dataframe)
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        corpus_dataframe["body"] = corpus_dataframe["body"].progress_apply(cleaner)

    
    data=corpus_dataframe.copy()
    
    # Create a dictionary to hold comments, grouped by rule and violation status
    data_by_rule_and_status = data.groupby(['rule', 'rule_violation'])['body'].apply(list).to_dict()
    
    
    # Create a list to hold the Triplet InputExample objects
    triplet_examples = []

    # Define the number of triplets to generate per violating comment
    NUM_TRIPLETS_PER_VIOLATION = 2 # <-- New Constant
    
    # Generate triplets
    for violating_row in data[data['rule_violation'] == True].itertuples():
        anchor_rule = violating_row.rule
        positive_comment = violating_row.body
        
        # Get the list of non-violating comments for the same rule
        # The .get() method is used to prevent errors if a rule has no non-violating comments
        # We use a default empty list in that case.
        non_violating_comments_for_rule = data_by_rule_and_status.get((anchor_rule, False), [])
        
        # Check if there are any non-violating comments for this rule
        if len(non_violating_comments_for_rule) >= NUM_TRIPLETS_PER_VIOLATION: # <-- Modified condition
            
            # Select NUM_TRIPLETS_PER_VIOLATION distinct non-violating comments
            negative_comments = random.sample(non_violating_comments_for_rule, NUM_TRIPLETS_PER_VIOLATION) # <-- Changed to random.sample
            
            # Create the TripletInputExample for each selected negative comment
            for negative_comment in negative_comments: # <-- Loop to create multiple triplets
                triplet_examples.append(InputExample(
                    texts=[anchor_rule, positive_comment, negative_comment]
                ))
        else:
            print(f"Skipping rule '{anchor_rule}' as less than {NUM_TRIPLETS_PER_VIOLATION} non-violating comments were found for it.") # <-- Modified print
    
    print(f"Generated {len(triplet_examples)} triplet examples.")
    print("\nExample Triplet:1")
    print(f"Anchor (Rule): {triplet_examples[0].texts[0]}")
    print(f"Positive (Violating Comment): {triplet_examples[0].texts[1]}")
    print(f"Negative (Non-Violating Comment): {triplet_examples[0].texts[2]}")

    print("\nExample Triplet:2")
    print(f"Anchor (Rule): {triplet_examples[1].texts[0]}")
    print(f"Positive (Violating Comment): {triplet_examples[1].texts[1]}")
    print(f"Negative (Non-Violating Comment): {triplet_examples[1].texts[2]}")

    
    #from sentence_transformers import SentenceTransformer, losses, util
    #from torch.utils.data import DataLoader
    
    # Assume you have your 'triplet_examples' list ready from the previous step.
    
    # --- Step 1: Load a Pre-trained Model ---
    # We recommend a model that is already good at semantic similarity tasks.
    model = SentenceTransformer(model_name_or_path=EMBDEDDING_MODEL_PATH,device="cuda",)
    print("Model loaded successfully.")

    
    # --- Step 2: Create a DataLoader ---
    # The DataLoader shuffles the examples and batches them for training.
    # A batch size of 16 or 32 is a good starting point for TripletLoss.
    train_dataloader = DataLoader(
        triplet_examples, 
        shuffle=True, 
        batch_size=8
    )
    print(f"DataLoader created with batch size: {train_dataloader.batch_size}")
    
    # --- Step 3: Define the TripletLoss ---
    # The TripletLoss class takes the model as an argument.
    # The 'margin' parameter is crucial. It ensures the distance between the
    # anchor and negative is at least 'margin' greater than the distance
    # between the anchor and the positive. A common value is 0.5 or 1.
    train_loss = losses.TripletLoss(model=model, triplet_margin=0.3,distance_metric=losses.TripletDistanceMetric.COSINE)
    print(f"TripletLoss initialized.")
    
    # --- Step 4: Fine-tune the Model ---
    # The model.fit() method handles the entire training loop.
    num_epochs = 1 # A small number of epochs is often enough for fine-tuning
    warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% of training steps
    
    print("\nStarting model fine-tuning...")
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=num_epochs,
        warmup_steps=warmup_steps,
        output_path=SAVE_MODEL_PATH, # The path to save your fine-tuned model
        show_progress_bar=True
    )
    
    print("\nFine-tuning complete. Model saved to './reddit-semantic-model'.")
    
    
    # --- Step 5: Verify the Fine-tuned Model (Optional) ---
    # You can now use the fine-tuned model to check if the embeddings
    # for your triplets are in the correct relative positions.
    
    # Example: Embed your first triplet
    anchor_embedding = model.encode(triplet_examples[0].texts[0])
    positive_embedding = model.encode(triplet_examples[0].texts[1])
    negative_embedding = model.encode(triplet_examples[0].texts[2])
    
    # Calculate distances
    dist_positive = util.pytorch_cos_sim(anchor_embedding, positive_embedding).item()
    dist_negative = util.pytorch_cos_sim(anchor_embedding, negative_embedding).item()
    
    print("\n--- Model Verification ---")
    print(f"Cosine Similarity (Anchor, Positive): {dist_positive:.4f}")
    print(f"Cosine Similarity (Anchor, Negative): {dist_negative:.4f}")
    
    # The model is working correctly if dist_positive > dist_negative.


Writing finetune_embedding_model.py


## Cell 5: `semantic.py` — Semantic Scoring

Writes the scoring script to disk. The `get_scores(test_dataframe, model_path)` function:
1. Loads the fine-tuned embedding model from `/kaggle/working/<model_path>`
2. Processes each unique rule independently (avoids cross-rule contamination)
3. Encodes both the test comments and the labeled corpus into L2-normalized embeddings
4. Runs **semantic search** (dot-product similarity = cosine similarity for normalized vectors, top-50 neighbors) to retrieve the most similar corpus examples for each test comment
5. **Signed scoring**: each test comment's score is `Σ(similarity × label)` where labels are `+1` (violating) and `-1` (non-violating). A positive net score means the comment is more similar to violations than to normal posts for that rule.

In [5]:
%%writefile semantic.py
import pandas as pd

#from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformer, losses, util
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm

from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE,CLEAN_TEXT

import pandas as pd
from sentence_transformers import InputExample
import random
from torch.utils.data import DataLoader
import os
from utils import cleaner

os.environ["WANDB_DISABLED"] = "true"

def get_scores(test_dataframe,SAVE_MODEL_PATH):
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
    embedding_model = SentenceTransformer(
        model_name_or_path="/kaggle/working/"+SAVE_MODEL_PATH,
        device="cuda",
    )

    result = []
    for rule in tqdm(test_dataframe["rule"].unique(), desc=f"Generate scores for each rule"):
        test_dataframe_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe_part.reset_index(names="row_id")
        
        query_embeddings = embedding_model.encode(
            sentences=test_dataframe_part["prompt"].tolist(),
            #prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        document_embeddings = embedding_model.encode(
            sentences=corpus_dataframe_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        test_dataframe_part["semantic"] = semantic_search(
            query_embeddings,
            document_embeddings,
            top_k=TOP_K,
            score_function=dot_score,
        )
        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            semantic = semantic.merge(
                corpus_dataframe_part[["row_id", "rule_violation"]],
                how="left",
                left_on="corpus_id",
                right_on="row_id",
            )
            semantic["score"] = semantic["score"]*semantic["rule_violation"]
            return semantic["score"].sum()
            
        tqdm.pandas(desc=f"Add label for {rule=}")
        test_dataframe_part["rule_violation"] = test_dataframe_part["semantic"].progress_apply(get_score)
        result.append(test_dataframe_part[["row_id", "rule_violation"]].copy())
        
    submission = pd.concat(result, axis=0)
    return submission


Writing semantic.py


## Cell 6: `generate_submission.py` — End-to-End Pipeline Script

Writes the top-level orchestration script. `generate_submission(base_model, save_path, output_file)`:
1. Loads and preprocesses the test set (prompt formatting + optional text cleaning)
2. Fine-tunes the given base embedding model and saves it locally (`train_embedding_model`)
3. Generates signed semantic violation scores for all test rows (`get_scores`)
4. Left-merges scores onto the full test index to preserve row ordering, then writes to CSV

When run as a CLI script, it accepts three positional arguments: `<base_model_path> <save_model_path> <output_filename>`.

In [6]:
%%writefile generate_submission.py

import pandas as pd
import sys 
from constants import DATA_PATH
from utils import get_dataframe_to_train, prepare_dataframe
from semantic import get_scores
from finetune_embedding_model import train_embedding_model

def generate_submission(EMBDEDDING_MODEL_PATH: str,SAVE_MODEL_PATH: str,OUTPUT_FILE: str):
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)

    #train_embedding_model("/kaggle/input/embeddingmodels-repo/downloaded_models/thenlper__gte-large/","reddit-semantic-model")
    #submission = get_scores(test_dataframe,"reddit-semantic-model")
    
    train_embedding_model(EMBDEDDING_MODEL_PATH,SAVE_MODEL_PATH)
    submission = get_scores(test_dataframe,SAVE_MODEL_PATH)
    submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
    submission.to_csv(OUTPUT_FILE, index=False)


if __name__ == "__main__":
    if len(sys.argv) < 4:
        print("Usage: python generate_submission.py <base_model> <saved_model> <output_filename>")
        sys.exit(1)
    
    EMBDEDDING_MODEL_PATH = sys.argv[1]
    SAVE_MODEL_PATH = sys.argv[2]
    OUTPUT_FILE = sys.argv[3]
    
    generate_submission(EMBDEDDING_MODEL_PATH,SAVE_MODEL_PATH,OUTPUT_FILE)

Writing generate_submission.py


## Cell 7: Run Pipeline — Model 1 (`thenlper/gte-large`)

Fine-tunes `gte-large` on the rule-violation corpus, saves the model as `reddit-semantic-model_m1`, and writes raw violation scores to `submission_m1.csv`.

In [7]:
! python generate_submission.py "/kaggle/input/embeddingmodels-repo/downloaded_models/thenlper__gte-large/" "reddit-semantic-model_m1" "submission_m1.csv"

2025-10-03 14:36:34.906342: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759502195.089733      57 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759502195.153884      57 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
cleaner: 100%|████████████████████████████| 1875/1875 [00:00<00:00, 5129.53it/s]
Generated 1944 triplet examples.

Example Triplet:1
Anchor (Rule): No legal advice: Do not offer or request legal advice.
Positive (Violating Comment): Lol. Try appealing the ban and say you won't do it again.
Negative (Non-Violating Comment): Morons in our government ... Huma was transcribing top secret and special access from the secure computer at sta

## Cell 8: Run Pipeline — Model 2 (`BAAI/bge-large-en-v1.5`)

Fine-tunes `bge-large-en-v1.5` on the same corpus, saves the model as `reddit-semantic-model_m2`, and writes raw violation scores to `submission_m2.csv`.

In [8]:
! python generate_submission.py "/kaggle/input/baai/transformers/bge-large-en-v1.5/1" "reddit-semantic-model_m2" "submission_m2.csv"

2025-10-03 14:41:42.444308: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759502502.467448      84 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759502502.475233      84 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
cleaner: 100%|████████████████████████████| 1875/1875 [00:00<00:00, 5316.73it/s]
Generated 1944 triplet examples.

Example Triplet:1
Anchor (Rule): No legal advice: Do not offer or request legal advice.
Positive (Violating Comment): Lol. Try appealing the ban and say you won't do it again.
Negative (Non-Violating Comment): Let her know she can give her extra money to me instead for taxes

Example Triplet:2
Anchor (Rule): No legal adv

## Cell 9: Run Pipeline — Model 3 (`intfloat/e5-large-v2`)

Fine-tunes `e5-large-v2` on the same corpus, saves the model as `reddit-semantic-model_m3`, and writes raw violation scores to `submission_m3.csv`.

In [9]:
! python generate_submission.py "/kaggle/input/embeddingmodels-repo/downloaded_models/intfloat__e5-large-v2/" "reddit-semantic-model_m3" "submission_m3.csv"

2025-10-03 14:45:29.475060: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759502729.497305     111 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759502729.504331     111 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
cleaner: 100%|████████████████████████████| 1875/1875 [00:00<00:00, 4677.57it/s]
Generated 1944 triplet examples.

Example Triplet:1
Anchor (Rule): No legal advice: Do not offer or request legal advice.
Positive (Violating Comment): Lol. Try appealing the ban and say you won't do it again.
Negative (Non-Violating Comment): hire private security to stay on the property and protect the sound set up you're going to rent and blast loud a

## Cell 10: Rank-Based Ensemble & Final Submission

Combines the three models' predictions into a single submission file:
1. **Rank-normalizes** each model's raw scores to `(0, 1)` using percentile rank — removes absolute scale differences between models so each contributes equally
2. **Averages** the three normalized scores (equal-weight ensemble)
3. Saves `submission.csv` with columns `row_id` and `rule_violation`

Rank normalization is preferred over raw score averaging because the three models produce scores on different absolute scales (different embedding dimensions, different training dynamics).

In [10]:
submission_emb1=pd.read_csv("/kaggle/working/submission_m1.csv")
submission_emb1['rule_violation_emb1'] = submission_emb1['rule_violation'].rank(method='average') / (len(submission_emb1)+1)

submission_emb2=pd.read_csv("/kaggle/working/submission_m2.csv")
submission_emb2['rule_violation_emb2'] = submission_emb2['rule_violation'].rank(method='average') / (len(submission_emb2)+1)

submission_emb3=pd.read_csv("/kaggle/working/submission_m3.csv")
submission_emb3['rule_violation_emb3'] = submission_emb3['rule_violation'].rank(method='average') / (len(submission_emb3)+1)

submission = submission_emb1[["row_id","rule_violation_emb1"]].merge(submission_emb2[["row_id","rule_violation_emb2"]], on="row_id", how="left")
submission = submission.merge(submission_emb3[["row_id","rule_violation_emb3"]], on="row_id", how="left")

submission['rule_violation']= (submission["rule_violation_emb1"]+submission["rule_violation_emb2"]+submission["rule_violation_emb3"])/3   
submission[["row_id","rule_violation"]].to_csv("submission.csv", index=False)

